# MedSigLIP Fine-tuning on CheXpert
Adapted from the ClinicalBERT + ResNeXt pipeline. Replaces both encoders with Google's MedSigLIP — a SigLIP-based 400M+400M vision-text model pretrained on diverse medical images including chest X-rays.

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoProcessor, AutoModel
from PIL import Image
import numpy as np
import os

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using device: MPS (Apple)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using device: CUDA (GPU) - {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("Using device: CPU")

In [ ]:
# Install dependencies
!pip install -q kaggle transformers accelerate

# --------- Run once ---------
# 1. Go to https://www.kaggle.com/<your-username>/account
# 2. Under 'API', click 'Create New API Token' to download kaggle.json
# 3. Upload kaggle.json to your Colab session then run:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

import kaggle

dataset_name = "ashery/chexpert"
download_path = "./CheXpert-v1.0-small"
kaggle.api.dataset_download_files(dataset_name, path=download_path, unzip=True)

print(f"Dataset downloaded to: {download_path}")
!ls -F {download_path}

### Load MedSigLIP
MedSigLIP is a gated model. You must:
1. Accept the [Health AI Developer Foundations terms of use](https://huggingface.co/google/medsiglip-448) on the HuggingFace model page
2. Create a HuggingFace token at https://huggingface.co/settings/tokens
3. In Colab: **Secrets** → add  with your token value

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Load your HF token from Colab secrets (or paste it directly)
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

MODEL_NAME = "google/medsiglip-448"

# AutoProcessor handles BOTH image preprocessing (resize to 448x448, normalize to [-1,1])
# AND text tokenization (max 64 tokens). Replaces the custom transforms + ClinicalBERT tokenizer.
processor = AutoProcessor.from_pretrained(MODEL_NAME, token=HF_TOKEN)

# Single unified model with .vision_model and .text_model sub-modules.
# Replaces both VisionEncoder (ResNeXt) and ClinicalTextEncoder (BioClinicalBERT).
model = AutoModel.from_pretrained(MODEL_NAME, token=HF_TOKEN).to(device)

print(f"Vision encoder params: {sum(p.numel() for p in model.vision_model.parameters()) / 1e6:.0f}M")
print(f"Text encoder params:   {sum(p.numel() for p in model.text_model.parameters()) / 1e6:.0f}M")
print(f"Embed dim:             {model.config.vision_config.hidden_size}")

### Data Loading

In [ ]:
train_full = pd.read_csv("/content/CheXpert-v1.0-small/train.csv")
val_full   = pd.read_csv("/content/CheXpert-v1.0-small/valid.csv")

# Fill NaN uncertainty labels with 0
train_full = train_full.fillna(0)
val_full   = val_full.fillna(0)

# Optional 50% subsets for faster iteration
train_subset = train_full.sample(frac=0.5, random_state=42)
val_subset   = val_full.sample(frac=0.5, random_state=42)

print(f"Train: {len(train_full)} | Val: {len(val_full)}")

### Report Generation
Same function as the original notebook — converts CheXpert labels into natural language radiology report strings.

In [ ]:
def generate_report_updated(row):
    labels = row.iloc[5:]  # Skip Path, Sex, Age, Frontal/Lateral, AP/PA

    positive_findings  = list(labels[labels == 1.0].index)
    uncertain_findings = list(labels[labels == -1.0].index)

    report_parts = []

    age  = int(row['Age'])            if pd.notna(row['Age'])            else None
    sex  = row['Sex'].lower()         if pd.notna(row['Sex'])            else None
    view = row['Frontal/Lateral'].lower() if pd.notna(row['Frontal/Lateral']) else None

    report_parts.append(f"{view.capitalize()} chest radiograph" if view else "Chest radiograph")

    demo = []
    if age: demo.append(f"{age}-year-old")
    if sex: demo.append(sex)
    if demo: report_parts.append(f"of {' '.join(demo)} patient")

    if not positive_findings and not uncertain_findings:
        report_parts.append("demonstrates no acute cardiopulmonary abnormality")
    else:
        findings_text = []
        if positive_findings:
            findings_text.append("shows " + ", ".join(f.lower().replace('_', ' ') for f in positive_findings))
        if uncertain_findings:
            findings_text.append("possible " + ", ".join(f.lower().replace('_', ' ') for f in uncertain_findings))
        report_parts.append(". ".join(findings_text))

    return " ".join(report_parts) + "."

### Dataset
Key difference from the original: we use  for both image and text preprocessing.
- Images are resized to **448×448** (MedSigLIP's native resolution) and normalized to **[-1, 1]**
- Text is tokenized with a max of **64 tokens** (MedSigLIP's context length limit)

No separate  tokenizer or custom  needed.

In [ ]:
class MedSigLIPDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        df = df.reset_index(drop=True)
        # Pre-generate all report strings
        self.reports   = df.apply(generate_report_updated, axis=1).tolist()
        self.img_paths = df["Path"].tolist()

    def __len__(self):
        return len(self.reports)

    def __getitem__(self, idx):
        report   = self.reports[idx]
        img_path = self.img_paths[idx]

        image = Image.open(img_path).convert("RGB")

        # processor handles BOTH modalities in one call.
        # padding="max_length" pads text to 64 tokens (MedSigLIP's context length).
        # The image is resized to 448x448 and normalised to [-1, 1] internally.
        encoded = processor(
            text=report,
            images=image,
            padding="max_length",
            max_length=64,
            truncation=True,
            return_tensors="pt",
        )

        return {
            "input_ids":      encoded["input_ids"].squeeze(0),       # (64,)
            "attention_mask": encoded["attention_mask"].squeeze(0),  # (64,)
            "pixel_values":   encoded["pixel_values"].squeeze(0),    # (3, 448, 448)
            "report":         report,
            "img_path":       img_path,
        }

### SigLIP Contrastive Loss
MedSigLIP was pretrained with the **sigmoid** (SigLIP) loss, which treats each image-text pair independently rather than normalising over the full batch like CLIP's softmax. This is the preferred loss for fine-tuning.

The model exposes  and  — learnable parameters that control the scaling and bias of the similarity logits, identical to the original SigLIP pretraining setup.

In [ ]:
def siglip_loss(image_embeds, text_embeds, logit_scale, logit_bias):
    """
    Sigmoid contrastive loss (SigLIP).
    Labels are +1 on the diagonal (matched pairs) and -1 everywhere else.
    Each pair is treated independently via sigmoid rather than softmax.
    """
    # Scaled dot-product similarity [B, B]
    logits = torch.matmul(image_embeds, text_embeds.T) * logit_scale.exp() + logit_bias

    n = logits.shape[0]
    # +1 for matched pairs (diagonal), -1 for mismatched pairs
    labels = 2.0 * torch.eye(n, device=logits.device) - 1.0

    # -log sigmoid(label * logit), summed and normalised
    loss = -F.logsigmoid(labels * logits).sum() / n
    return loss

### Recall@K (unchanged from original)

In [ ]:
def recall_at_k(image_embeds, text_embeds, k_values=[1, 5]):
    similarity = torch.matmul(image_embeds, text_embeds.T)
    results = {}

    for k in k_values:
        top_k = similarity.topk(k, dim=1).indices
        correct = torch.tensor([i in top_k[i] for i in range(len(image_embeds))], dtype=torch.float)
        results[f"image_to_text_recall@{k}"] = correct.mean().item()

    for k in k_values:
        top_k = similarity.T.topk(k, dim=1).indices
        correct = torch.tensor([i in top_k[i] for i in range(len(text_embeds))], dtype=torch.float)
        results[f"text_to_image_recall@{k}"] = correct.mean().item()

    return results

In [ ]:
train_dataset = MedSigLIPDataset(train_full)
val_dataset   = MedSigLIPDataset(val_full)

# Batch size 16 is safer for the larger 448x448 inputs on a T4 GPU.
# Increase to 32 if you have an A100.
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

SAVE_PATH = "/content/drive/MyDrive/models/medsiglip_chexpert/"
os.makedirs(SAVE_PATH, exist_ok=True)

### Training Loop
**Differentiated learning rates** across three parameter groups, mirroring the original notebook's approach:
-  backbone — low LR (already pretrained on chest X-rays)
-  backbone — low LR
-  /  — slightly higher LR (same role as the projection layers in the original)

The training loop is otherwise identical in structure to the original: AMP, periodic validation, periodic checkpointing.

In [ ]:
# =========================
# Hyperparameters
# =========================
max_epoch_number = 4
test_freq  = 500   # validate every N iterations (halved because batch size halved)
ckpt_freq  = 500   # checkpoint every N iterations

# Differentiated learning rates (mirrors the original notebook's approach)
vision_lr = 5e-6   # vision backbone (lower than original — MedSigLIP already knows CXRs)
text_lr   = 5e-6   # text backbone
scale_lr  = 1e-4   # logit_scale / logit_bias (analogous to the projection LR)

optimizer = torch.optim.Adam([
    {'params': model.vision_model.parameters(), 'lr': vision_lr},
    {'params': model.text_model.parameters(),   'lr': text_lr},
    {'params': [model.logit_scale, model.logit_bias], 'lr': scale_lr},
])

scaler = torch.cuda.amp.GradScaler()

# =========================
# Training
# =========================
iteration = 0
for epoch in range(max_epoch_number):
    model.train()
    batch_losses = []

    for batch in train_loader:
        pixel_values  = batch["pixel_values"].to(device)
        input_ids     = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
            )
            # outputs.image_embeds and text_embeds are already L2-normalised
            image_embeds = outputs.image_embeds
            text_embeds  = outputs.text_embeds

            if torch.isnan(image_embeds).any():
                print("NaN in image_embeds"); break
            if torch.isnan(text_embeds).any():
                print("NaN in text_embeds"); break

            loss = siglip_loss(image_embeds, text_embeds, model.logit_scale, model.logit_bias)

            if torch.isnan(loss).any():
                print("NaN in loss"); break

        batch_loss_value = loss.item()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_losses.append(batch_loss_value)

        # =========================
        # Validation
        # =========================
        if iteration % test_freq == 0:
            model.eval()
            val_losses = []
            all_img_embeds  = []
            all_text_embeds = []

            with torch.no_grad():
                for batch_val in val_loader:
                    val_pixels = batch_val["pixel_values"].to(device)
                    val_ids    = batch_val["input_ids"].to(device)
                    val_mask   = batch_val["attention_mask"].to(device)

                    with torch.cuda.amp.autocast():
                        val_outputs = model(
                            pixel_values=val_pixels,
                            input_ids=val_ids,
                            attention_mask=val_mask,
                        )
                        val_img_e  = val_outputs.image_embeds
                        val_txt_e  = val_outputs.text_embeds
                        val_loss   = siglip_loss(val_img_e, val_txt_e, model.logit_scale, model.logit_bias)

                    val_losses.append(val_loss.item())
                    all_img_embeds.append(val_img_e.cpu().float())
                    all_text_embeds.append(val_txt_e.cpu().float())

            all_img_embeds  = torch.cat(all_img_embeds,  dim=0)
            all_text_embeds = torch.cat(all_text_embeds, dim=0)

            avg_val_loss = float(np.mean(val_losses))
            recall_results = recall_at_k(all_img_embeds, all_text_embeds)
            rounded = {k: round(v, 5) for k, v in recall_results.items()}
            print(f"epoch:{epoch+1:2d} iter:{iteration:4d} val loss:{avg_val_loss:.3f}  recall@k:{rounded}")

            model.train()

        # =========================
        # Periodic checkpoint
        # =========================
        if iteration % ckpt_freq == 0:
            ckpt_path = os.path.join(SAVE_PATH, f"checkpoint_iter_{iteration}.pt")
            torch.save({
                "epoch":       epoch,
                "iteration":   iteration,
                "model":       model.state_dict(),
                "optimizer":   optimizer.state_dict(),
                "scaler":      scaler.state_dict(),
                "train_loss":  float(batch_loss_value),
            }, ckpt_path)
            print(f"Checkpoint saved to {ckpt_path}")

        iteration += 1

    train_loss = float(np.mean(batch_losses))
    print(f"epoch:{epoch+1:2d} iter:{iteration:4d} train loss:{train_loss:.3f}
")

# =========================
# Final checkpoint
# =========================
final_path = os.path.join(SAVE_PATH, "final_checkpoint_medsiglip.pt")
torch.save({
    "epoch":     epoch,
    "iteration": iteration,
    "model":     model.state_dict(),
    "optimizer": optimizer.state_dict(),
    "scaler":    scaler.state_dict(),
    "train_loss": train_loss,
}, final_path)
print(f"Final checkpoint saved to {final_path}")

### Inference — Image-to-Image Retrieval
At retrieval time, only the **vision encoder** is used. Build an embedding index over the database, then find nearest neighbours by cosine similarity.

In [ ]:
@torch.no_grad()
def embed_images(img_paths, batch_size=32):
    """
    Embed a list of image paths using MedSigLIP's vision encoder.
    Returns a (N, D) float32 tensor of L2-normalised embeddings.
    """
    model.eval()
    all_embeds = []

    for start in range(0, len(img_paths), batch_size):
        batch_paths = img_paths[start : start + batch_size]
        images = [Image.open(p).convert("RGB") for p in batch_paths]

        # processor auto-resizes to 448x448 and normalises to [-1, 1]
        inputs = processor(images=images, return_tensors="pt").to(device)

        with torch.cuda.amp.autocast():
            vision_outputs = model.vision_model(**inputs)
            # Pool + project to get the final embedding (same path as the full forward)
            embeds = model.visual_projection(vision_outputs.pooler_output)
            embeds = F.normalize(embeds, p=2, dim=-1)

        all_embeds.append(embeds.cpu().float())

    return torch.cat(all_embeds, dim=0)


def retrieve_similar(query_path, db_paths, db_embeds, top_k=5):
    """
    Given a query image path and a pre-computed embedding database,
    return the top-k most similar images.
    """
    query_embed = embed_images([query_path])  # (1, D)
    similarities = (query_embed @ db_embeds.T).squeeze(0)  # (N,)
    top_k_indices = similarities.topk(top_k).indices.tolist()

    return [(db_paths[i], similarities[i].item()) for i in top_k_indices]


# --- Example usage ---
# db_paths = val_dataset.img_paths
# db_embeds = embed_images(db_paths)
# torch.save(db_embeds, os.path.join(SAVE_PATH, "val_image_embeddings.pt"))
#
# results = retrieve_similar(query_path=db_paths[0], db_paths=db_paths, db_embeds=db_embeds, top_k=5)
# for path, score in results:
#     print(f"{score:.4f}  {path}")